# Rental crisis in Portugal — H2

**Hypothesis H2:** municipalities with more short-term rentals (STR) per 1,000 residents have a higher median rent (€/m²).

| Data | Source | How we get it |
|---|---|---|
| Median rent €/m² per municipality | INE (Statistics Portugal) | **API** |
| Short-term rental registrations | RNAL – Turismo de Portugal | **Dataset** (CSV download) |
| Population per municipality | INE (Statistics Portugal) | **API** (same function as the rent) |

**Steps:** 1. Setup → 2. Collect data → 3. Understand the data → 4. Clean → 5. Combine → 6. Analyse → 7. Conclusions

## 1. Setup

In [ ]:
import time
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

In [ ]:
# Folders (this notebook lives in notebooks/, the data folder is one level up)
RAW_FOLDER = Path("../data/raw")
CLEAN_FOLDER = Path("../data/clean")
FIGURES_FOLDER = Path("../figures")
for folder in [RAW_FOLDER, CLEAN_FOLDER, FIGURES_FOLDER]:
    folder.mkdir(parents=True, exist_ok=True)

# We tell the websites who we are (good practice when collecting data)
HEADERS = {"User-Agent": "data-wrangling-project (student project)"}

## 2. Data collection

### 2.1 INE API — median rent €/m² and population

* Documentation: [INE API](https://www.ine.pt/xportal/xmain?xpid=INE&xpgid=ine_api_db)
* Endpoint: `https://www.ine.pt/ine/json_indicador/pindica.jsp?op=2&varcd=<code>&lang=PT`
* INE does not publish a rate limit, so we are polite: we wait 1 second after each request and, if the server answers **429 (too many requests)**, we wait and try again.
* If we don't choose a period (`Dim1`), the API returns the **most recent period** available.
* We save the result in `data/raw/` so we only call the API once.

| Indicator | Code | Latest period (Sept 2026) |
|---|---|---|
| Median rent €/m² of new leases, last 12 months (Methodology 2026, NUTS 2024, quarterly) | `0014696` | 2nd quarter 2026 |
| Resident population by municipality, sex and age (NUTS 2024, annual) | `0012918` | 2025 |

⚠️ Older codes (`0009817`, `0008273`) use the old regions (NUTS 2013) and stopped in 2023.

In [ ]:
INE_URL = "https://www.ine.pt/ine/json_indicador/pindica.jsp"
RENT_INDICATOR = "0014696"         # median rent €/m² of new leases, last 12 months (NUTS 2024, quarterly)
POPULATION_INDICATOR = "0012918"   # resident population by municipality, sex and age (NUTS 2024, annual)


def get_json(url, params):
    """Call an API and return the JSON. Retries up to 3 times if the server is busy (HTTP 429)."""
    for attempt in range(1, 4):
        response = requests.get(url, params=params, headers=HEADERS, timeout=60)
        if response.status_code == 429:
            print(f"Too many requests – waiting {attempt * 5} seconds...")
            time.sleep(attempt * 5)
            continue
        response.raise_for_status()   # stops with an error for other problems (404, 500...)
        time.sleep(1)                 # be polite before the next request
        return response.json()
    raise Exception("The API is still busy after 3 attempts.")


def ine_json_to_table(ine_json):
    """INE returns {'Dados': {period: [records]}}. Turn it into one row per period and place."""
    rows = []
    for period, records in ine_json[0]["Dados"].items():
        for record in records:
            record["period"] = period
            rows.append(record)
    return pd.DataFrame(rows)

In [ ]:
rent_file = RAW_FOLDER / "ine_rent_raw.csv"

if rent_file.exists():
    print("Reading the saved file")
    rent_raw = pd.read_csv(rent_file, dtype=str)
else:
    print("Calling the INE API")
    ine_json = get_json(INE_URL, {"op": "2", "varcd": RENT_INDICATOR, "lang": "PT"})
    rent_raw = ine_json_to_table(ine_json)
    rent_raw.to_csv(rent_file, index=False)

rent_raw.head()

### 2.2 RNAL dataset — short-term rental registrations

1. Go to the [Turismo de Portugal open-data portal](https://dadosabertos.turismodeportugal.pt/datasets/estabelecimentos-de-alojamento-local).
2. Download the dataset as **CSV**.
3. Save it as `data/raw/rnal.csv`.

In [ ]:
rnal_raw = pd.read_csv(RAW_FOLDER / "rnal.csv", sep=None, engine="python", dtype=str)  # sep=None detects ; or ,
rnal_raw.head()

In [ ]:
population_file = RAW_FOLDER / "ine_population_raw.csv"

if population_file.exists():
    print("Reading the saved file")
    population_raw = pd.read_csv(population_file, dtype=str)
else:
    print("Calling the INE API")
    # No Dim1 -> the most recent year. Dim3 = sex, Dim4 = age group ("T" = total of all)
    params = {"op": "2", "varcd": POPULATION_INDICATOR, "lang": "PT", "Dim3": "T", "Dim4": "T"}
    population_raw = ine_json_to_table(get_json(INE_URL, params))
    population_raw.to_csv(population_file, index=False)

print("Population year:", population_raw["period"].iloc[0])
population_raw.head()

## 3. Understanding the data

Before cleaning: how many rows, which columns, missing values and duplicates?

In [ ]:
def quick_check(df, name):
    print(f"--- {name} ---")
    print("Rows, columns:", df.shape)
    print("Duplicated rows:", df.duplicated().sum())
    print("Missing values per column:")
    print(df.isna().sum())
    print()


quick_check(rent_raw, "INE rent")
quick_check(rnal_raw, "RNAL")
quick_check(population_raw, "INE population")

In [ ]:
# Look at the column names of RNAL – we need the one with the municipality ("Concelho")
print(rnal_raw.columns.tolist())

In [ ]:
# INE mixes several geographic levels. The length of 'geocod' tells us which one:
# 'PT' = Portugal, 2-3 characters = regions, 7 characters = municipality
print(rent_raw["geocod"].str.len().value_counts())
print("Periods:", rent_raw["period"].unique())

**What we found and our cleaning plan**

| Problem | Cleaning technique |
|---|---|
| Column names with spaces, capitals and symbols | rename columns |
| Columns we don't need | drop columns |
| INE has countries/regions and several periods | filter rows (municipalities, latest period) |
| Values stored as text (`"6.49"`, `"567131"`) | convert to numbers |
| Names written differently in RNAL and INE (`LISBOA` vs `Lisboa`, `Evora` vs `Évora`) | string manipulation to one standard name |
| Missing values and duplicates | drop them |
| Azores and Madeira (e.g. *Lagoa* exists twice) | keep mainland Portugal only |

## 4. Cleaning

### 4.1 Small helper function used by all datasets

In [ ]:
def standard_name(name):
    """Make names comparable: 'ÉVORA ' -> 'evora'."""
    name = str(name).strip().lower()
    name = unicodedata.normalize("NFKD", name)                    # split 'é' into 'e' + accent
    name = "".join(letter for letter in name if not unicodedata.combining(letter))  # drop the accents
    return name

### 4.2 INE rent

In [ ]:
def keep_mainland_municipalities(df):
    """Municipalities have a 7-character geocod; mainland Portugal codes start with '1'."""
    return df[(df["geocod"].str.len() == 7) & (df["geocod"].str.startswith("1"))]


def keep_latest_period(df):
    """Keep only the most recent period (latest year, then latest semester)."""
    df = df.copy()
    df["year"] = df["period"].str.extract(r"(\d{4})", expand=False)
    latest_year_periods = df[df["year"] == df["year"].max()]["period"].unique()
    latest_period = sorted(latest_year_periods)[-1]
    print("Latest period:", latest_period)
    return df[df["period"] == latest_period]


def clean_rent(df):
    """Main cleaning function for the INE rent data."""
    df = keep_mainland_municipalities(df)
    df = keep_latest_period(df)
    df = df.rename(columns={"geodsg": "municipality", "valor": "median_rent_m2"}).copy()
    df["median_rent_m2"] = pd.to_numeric(df["median_rent_m2"], errors="coerce")  # text -> number
    df = df.dropna(subset=["median_rent_m2"])                                   # no value = no use
    df["name_key"] = df["municipality"].apply(standard_name)
    return df[["geocod", "municipality", "name_key", "median_rent_m2"]]         # drop other columns


rent = clean_rent(rent_raw)
print(rent.shape)
rent.head()

### 4.3 RNAL registrations

⚠️ Change `RNAL_MUNICIPALITY_COLUMN` if the column has a different name in your file (see the list printed in section 3).

**Limitation:** RNAL counts *registrations*. Some places may no longer be active, so our number is an upper limit of the real supply. If the file has a status column (e.g. *Estado*), we could keep only the active ones.

**Check:** if the file includes Azores and Madeira, the Azores *Lagoa* will be counted together with Lagoa in the Algarve. If there is a district/region column, remove the Azores and Madeira rows first.

In [ ]:
RNAL_MUNICIPALITY_COLUMN = "Concelho"


def clean_rnal(df):
    """Main cleaning function for the RNAL data: one row per registration."""
    df = df.drop_duplicates().copy()                            # same registration twice
    df = df.dropna(subset=[RNAL_MUNICIPALITY_COLUMN])           # registrations without municipality
    df["name_key"] = df[RNAL_MUNICIPALITY_COLUMN].apply(standard_name)
    return df[["name_key"]]


rnal = clean_rnal(rnal_raw)
print(rnal.shape)
rnal.head()

### 4.4 INE population

In [ ]:
def keep_totals(df):
    """Keep only the rows for 'all sexes' and 'all ages' (code 'T'), if those columns exist."""
    for column in ["dim_3", "dim_4"]:
        if column in df.columns:
            df = df[df[column] == "T"]
    return df


def clean_population(df):
    """Main cleaning function for the INE population data."""
    df = keep_mainland_municipalities(df)     # same function as for the rent
    df = keep_totals(df)
    df = df.rename(columns={"valor": "population"}).copy()
    df["population"] = pd.to_numeric(df["population"], errors="coerce")
    df = df.dropna(subset=["population"])
    df = df.drop_duplicates(subset="geocod")
    return df[["geocod", "population"]]


population = clean_population(population_raw)
print(population.shape)
population.head()

## 5. Combining the data

1. **Group** RNAL: count the registrations per municipality.
2. **Merge**: rent + population on the INE code `geocod` (same source, so the codes match exactly), then RNAL on the standardised name `name_key`.
3. **New field:** STR registrations per 1,000 residents.

In [ ]:
str_count = rnal.groupby("name_key").size().reset_index(name="str_registrations")
str_count.sort_values("str_registrations", ascending=False).head(10)

In [ ]:
h2 = rent.merge(population, on="geocod", how="inner")
h2 = h2.merge(str_count, on="name_key", how="left")
h2["str_registrations"] = h2["str_registrations"].fillna(0)   # not in RNAL = 0 registrations
h2["str_per_1000"] = h2["str_registrations"] / h2["population"] * 1000

print("Municipalities with rent data:", len(rent))
print("Municipalities after merging:", len(h2))

In [ ]:
# Which RNAL names did not match an INE municipality? (spelling differences show up here)
not_matched = set(str_count["name_key"]) - set(rent["name_key"])
print("In RNAL but not in INE:", not_matched)

In [ ]:
h2.to_csv(CLEAN_FOLDER / "h2_clean.csv", index=False)
h2.sort_values("str_per_1000", ascending=False).head(10)

## 6. Analysis

### 6.1 Split municipalities into 4 groups by STR density

In [ ]:
h2["str_group"] = pd.qcut(h2["str_per_1000"].rank(method="first"), q=4,
                          labels=["Low", "Medium-low", "Medium-high", "High"])

rent_by_group = h2.groupby("str_group", observed=True)["median_rent_m2"].median()
rent_by_group

In [ ]:
rent_by_group.plot(kind="bar", color="steelblue", rot=0)
plt.title("Median rent by short-term rental group")
plt.xlabel("STR registrations per 1,000 residents (4 groups)")
plt.ylabel("Median rent (€/m²)")
plt.tight_layout()
plt.savefig(FIGURES_FOLDER / "h2_rent_by_group.png")
plt.show()

### 6.2 Correlation and scatter plot

In [ ]:
correlation = h2["str_per_1000"].corr(h2["median_rent_m2"], method="spearman")
print(f"Spearman correlation: {correlation:.2f}")
# Spearman works with rankings, so a few extreme municipalities (e.g. Albufeira, Lisbon) matter less.
# Rule of thumb: < 0.2 very weak | 0.2–0.4 weak | 0.4–0.6 moderate | > 0.6 strong

In [ ]:
plt.scatter(h2["str_per_1000"], h2["median_rent_m2"], alpha=0.6)
plt.xscale("symlog")  # log-like scale that also accepts 0; spreads the points because a few values are very high
plt.title(f"STR density vs. median rent (Spearman = {correlation:.2f})")
plt.xlabel("STR registrations per 1,000 residents (log scale)")
plt.ylabel("Median rent (€/m²)")
plt.tight_layout()
plt.savefig(FIGURES_FOLDER / "h2_scatter.png")
plt.show()

### 6.3 Filtering: top 10 municipalities with the most short-term rentals

In [ ]:
top_10 = h2.sort_values("str_per_1000", ascending=False).head(10)
top_10[["municipality", "str_per_1000", "median_rent_m2"]].round(2)

In [ ]:
# Is the result only caused by these 10 places? Remove them and calculate again.
without_top_10 = h2[~h2["name_key"].isin(top_10["name_key"])]
correlation_without_top_10 = without_top_10["str_per_1000"].corr(without_top_10["median_rent_m2"], method="spearman")
print(f"Spearman correlation without the top 10: {correlation_without_top_10:.2f}")

## 7. Conclusions

*Write this part after running the notebook.*

* **Result:** the Spearman correlation is ___, so the relationship is ___. The "High" group pays ___ €/m² vs ___ €/m² in the "Low" group.
* **Is H2 supported?** ___
* **Correlation is not causation.** Places can be touristy *and* expensive for other reasons: coast and climate, jobs and wages (Lisbon, Porto), little new housing, foreign residents.
* **Limitations:** RNAL counts registrations (not only active places); INE has no rent value for some small municipalities; the data sources are from different dates.
* **Next steps:** compare municipalities within the same district; look at how STR and rents grew over time.